In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

iris = load_iris()
mask = (iris.target == 0) | (iris.target == 1)
X = iris.data[mask][:, :2]
y = iris.target[mask]
nombres = iris.target_names[:2]

plt.figure(figsize=(8, 6))
for clase, nombre in enumerate(nombres):
    plt.scatter(X[y == clase, 0], X[y == clase, 1], label=nombre, alpha=0.7)
plt.xlabel("Longitud del sépalo (cm)")
plt.ylabel("Ancho del sépalo (cm)")
plt.title("Iris: dos clases y dos características")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)

In [ ]:
np.random.seed(0)


class RedNeuronal:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.pesos1 = np.random.rand(4)   # [wa1, wb1, wc1, wd1]
        self.sesgos1 = np.random.rand(2)  # [b11, b12]
        self.pesos2 = np.random.rand(2)   # [wa2, wb2]
        self.sesgos2 = np.random.rand(1)  # [b21]
        self.errores = []

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def forward(self, x1, x2):
        wa1, wb1, wc1, wd1 = self.pesos1
        b11, b12 = self.sesgos1
        wa2, wb2 = self.pesos2
        b21 = self.sesgos2[0]

        z11 = wa1 * x1 + wc1 * x2 + b11
        z12 = wb1 * x1 + wd1 * x2 + b12
        a11 = self.sigmoid(z11)
        a12 = self.sigmoid(z12)
        z21 = wa2 * a11 + wb2 * a12 + b21
        y_gorro = self.sigmoid(z21)
        return a11, a12, y_gorro

    def gradientes(self, x1, x2, y):
        a11, a12, y_gorro = self.forward(x1, x2)
        error = 0.5 * (y - y_gorro) ** 2

        C = (y_gorro - y) * y_gorro * (1 - y_gorro)  # delta_1[2]
        A = C * self.pesos2[0] * a11 * (1 - a11)    # delta_1[1]
        B = C * self.pesos2[1] * a12 * (1 - a12)    # delta_2[1]

        grad_pesos1 = np.array([A * x1, B * x1, A * x2, B * x2])
        grad_sesgos1 = np.array([A, B])
        grad_pesos2 = np.array([C * a11, C * a12])
        grad_sesgos2 = np.array([C])
        return error, grad_pesos1, grad_sesgos1, grad_pesos2, grad_sesgos2

    def entrenamiento(self, tasa_aprendizaje=0.1, epocas=1000):
        self.errores = []
        for k in range(epocas):
            for i in range(self.x.shape[0]):
                x1, x2 = self.x[i]
                _, gp1, gs1, gp2, gs2 = self.gradientes(x1, x2, self.y[i])

                # Todos los gradientes se calculan antes de actualizar parámetros.
                self.pesos1 -= tasa_aprendizaje * gp1
                self.sesgos1 -= tasa_aprendizaje * gs1
                self.pesos2 -= tasa_aprendizaje * gp2
                self.sesgos2 -= tasa_aprendizaje * gs2

            _, _, predicciones = self.forward(self.x[:, 0], self.x[:, 1])
            error = np.mean(0.5 * (self.y - predicciones) ** 2)
            self.errores.append(float(error))
            if (k + 1) % 100 == 0:
                print(f"Época {k + 1}: pérdida = {error:.6f}")
        return self.errores

    def clasificacion(self, x1, x2):
        _, _, y_gorro = self.forward(x1, x2)
        return int(y_gorro >= 0.5)


red_neuronal = RedNeuronal(X_train, y_train)
errores = red_neuronal.entrenamiento()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(errores) + 1), errores)
plt.xlabel("Época")
plt.ylabel("Pérdida cuadrática media (con factor 1/2)")
plt.title("Pérdida de entrenamiento al finalizar cada época")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
y_pred = np.array([red_neuronal.clasificacion(x[0], x[1]) for x in X_test])
print(f"Accuracy en test: {accuracy_score(y_test, y_pred):.2%}")

matriz = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(matriz, display_labels=nombres).plot(cmap="Blues")
plt.title("Matriz de confusión: test")
plt.show()

In [ ]:
x1_grid = np.linspace(X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5, 250)
x2_grid = np.linspace(X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5, 250)
xx, yy = np.meshgrid(x1_grid, x2_grid)
_, _, probabilidades = red_neuronal.forward(xx, yy)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, probabilidades >= 0.5, levels=[-0.5, 0.5, 1.5],
             colors=["tab:blue", "tab:orange"], alpha=0.15)
plt.contour(xx, yy, probabilidades, levels=[0.5], colors="black", linewidths=1)
for clase, nombre in enumerate(nombres):
    plt.scatter(X_test[y_test == clase, 0], X_test[y_test == clase, 1],
                color=["tab:blue", "tab:orange"][clase], label=nombre)
fallos = y_pred != y_test
if fallos.any():
    plt.scatter(X_test[fallos, 0], X_test[fallos, 1], s=160,
                facecolors="none", edgecolors="red", linewidths=2, label="Error")
plt.xlabel("Longitud del sépalo (estandarizada)")
plt.ylabel("Ancho del sépalo (estandarizado)")
plt.title("Frontera de decisión e instancias de test")
plt.legend()
plt.grid(alpha=0.3)
plt.show()